# Graph-Based News Intelligence Engine
This notebook connects directly to your live Neo4j database instance. 

In [1]:
# 1. Dependencies and Driver Setup
import pandas as pd
from neo4j import GraphDatabase

# Connection configurations matching your Version 2 setup
URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "DMPrj212345") 

driver = GraphDatabase.driver(URI, auth=AUTH)
print("Neo4j Database Connectivity Established Successfully!")

Neo4j Database Connectivity Established Successfully!


<h2>ETL Pipeline: News Graph Construction</h2>

<p>
This pipeline transforms raw news and user behavior data into a Neo4j graph for analytics and recommendations.
</p>

<h3>1. Overview</h3>
<p>
The system processes news articles, users, topics, entities, and sources, and builds relationships for personalization and semantic analysis.
</p>

<h3>2. Article Processing</h3>
<p>
Each article is stored as a node with title, abstract, category, URL, and a sentence embedding. Topics are extracted from categories.
</p>
<ul>
  <li>(Article)-[:HAS_TOPIC]->(Topic)</li>
  <li>(Article)-[:FROM]->(Source)</li>
  <li>(Source)-[:PUBLISHED]->(Article)</li>
  <li>(Article)-[:MENTIONS]->(Entity)</li>
</ul>

<h3>3. Source Handling</h3>
<p>
Article sources are extracted from URLs. If unavailable, synthetic sources (e.g., BBC, CNN, Reuters) are used to ensure completeness.
</p>

<h3>4. User Behavior</h3>
<p>
Users are created as Subscriber nodes with reading history and synthetic attributes like frequency and email.
</p>
<ul>
  <li>(Subscriber)-[:READ]->(Article)</li>
  <li>(Subscriber)-[:FOLLOWS]->(Source)</li>
</ul>

<h3>5. Interest Inference</h3>
<p>
Users are linked to topics they read frequently (more than or equal to 2 articles).
</p>
<ul>
  <li>(Subscriber)-[:INTERESTED_IN]->(Topic)</li>
</ul>

<h3>6. Outcome</h3>
<p>
The final graph enables recommendation systems, semantic search, and user personalization using embeddings and relationships.
</p>

In [2]:
Comment this line to execute the code. This is a pipeline for data preprocessing, data uploading pipeline. This creates database in neo4j
    
import json
import random
import numpy as np
import pandas as pd
from urllib.parse import urlparse
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# Seed for reproducibility of synthetic generation
random.seed(42)

# 1. Initialize Embeddings Model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Ground truth synthetic pool fallback list
GLOBAL_SOURCES = [
    {"name": "BBC News", "url": "https://bbc.com"},
    {"name": "CNN", "url": "https://cnn.com"},
    {"name": "Reuters", "url": "https://reuters.com"},
    {"name": "The New York Times", "url": "https://nytimes.com"},
    {"name": "The Guardian", "url": "https://theguardian.com"},
    {"name": "Bloomberg", "url": "https://bloomberg.com"},
    {"name": "ESPN", "url": "https://espn.com"},
    {"name": "TechCrunch", "url": "https://techcrunch.com"}
]

def clean_source_mapping(url_string, article_id):
    """
    Extracts actual domain names from real data URLs when possible,
    otherwise matches down to a rich synthetic source group pool.
    """
    if url_string and "msn.com" not in url_string:
        try:
            parsed = urlparse(url_string)
            domain = parsed.netloc.replace("www.", "")
            if domain:
                return domain.split(".")[0].upper(), f"https://{domain}"
        except Exception:
            pass
            
    # Fallback/Synthetic distribution based on numerical ID string tracking
    try:
        num_id = int(''.join(filter(str.isdigit, article_id)))
    except ValueError:
        num_id = len(article_id)
        
    selected = GLOBAL_SOURCES[num_id % len(GLOBAL_SOURCES)]
    return selected["name"], selected["url"]


def run_etl(news_path, behaviors_path):
    driver = GraphDatabase.driver(URI, auth=AUTH)

    # -------------------------------------------------------------
    # PHASE 1: Process and Upload Articles, Topics, Entities, Sources
    # -------------------------------------------------------------
    print("Processing News Items and Publishers...")
    news_cols = [
        "nid", "category", "subcategory", "title", "abstract", 
        "url", "title_entities", "abstract_entities"
    ]
    df_news = pd.read_csv(news_path, sep="\t", names=news_cols, keep_default_na=False)

    with driver.session() as session:
        # Enforce Explicit Database Constraints
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (a:Article) REQUIRE a.id IS UNIQUE")
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (t:Topic) REQUIRE t.id IS UNIQUE")
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (e:Entity) REQUIRE e.id IS UNIQUE")
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (s:Source) REQUIRE s.id IS UNIQUE")

        # Set up a dictionary pool of unique sources created during execution for user mapping later
        source_ids_pool = set()

        print("Writing Articles and Structural Schema Elements to Graph...")
        for _, row in tqdm(df_news.iterrows(), total=len(df_news)):
            text_to_embed = f"{row['title']}. {row['abstract']}"
            embedding = model.encode(text_to_embed).tolist()

            # Process / Synthesize exact Publisher data details
            src_name, src_url = clean_source_mapping(row["url"], row["nid"])
            src_id = src_name.lower().replace(" ", "_").replace(".", "_")
            source_ids_pool.add(src_id)

            cypher_article = """
            MERGE (a:Article {id: $nid})
            SET a.title = $title,
                a.content = $abstract,
                a.url = $url,
                a.embedding = $embedding,
                a.published_at = datetime()
            
            // Topic Nodes
            MERGE (t:Topic {id: $category})
            SET t.name = $category,
            MERGE (a)-[:HAS_TOPIC]->(t)
            
            // Source Nodes
            MERGE (src:Source {id: $src_id})
            SET src.name = $src_name,
                src.url = $src_url
            MERGE (src)-[:PUBLISHED]->(a)
            MERGE (a)-[:FROM]->(src)
            """
            
            session.run(
                cypher_article,
                nid=row["nid"], title=row["title"], abstract=row["abstract"],
                url=row["url"], category=row["category"], embedding=embedding,
                pol_mapping=pol_mapping, src_id=src_id, src_name=src_name, src_url=src_url
            )

            # Process Entity Arrays strings
            for entity_col in ["title_entities", "abstract_entities"]:
                if row[entity_col]:
                    try:
                        entities = json.loads(row[entity_col])
                        for ent in entities:
                            session.run(
                                """
                                MERGE (e:Entity {id: $ent_id})
                                SET e.name = $label, e.type = $type
                                WITH e
                                MATCH (a:Article {id: $nid})
                                MERGE (a)-[:MENTIONS]->(e)
                                """,
                                ent_id=ent["WikidataId"], label=ent["Label"],
                                type=ent["Type"], nid=row["nid"]
                            )
                    except Exception:
                        continue

    # Convert structural source identities to list format for array selection operations
    source_ids_list = list(source_ids_pool)

    # -------------------------------------------------------------
    # PHASE 2: Upload Subscribers & Map Personalization Layers
    # -------------------------------------------------------------
    print("Processing Subscriber Behaviors and Interest Alignments...")
    behavior_cols = ["impression_id", "uid", "time", "history", "impressions"]
    df_behaviors = pd.read_csv(behaviors_path, sep="\t", names=behavior_cols, keep_default_na=False)
    unique_users = df_behaviors[["uid", "history"]].drop_duplicates(subset=["uid"])

    with driver.session() as session:
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (s:Subscriber) REQUIRE s.id IS UNIQUE")

        for _, row in tqdm(unique_users.iterrows(), total=len(unique_users)):
            # Establish Subscriber Base Node
            freq = random.choice(["daily", "weekly", "monthly"])
            session.run(
                """
                MERGE (s:Subscriber {id: $uid})
                SET s.name = 'User_' + $uid, 
                    s.email = $uid + '@example.com',
                    s.frequency = $freq
                """,
                uid=row["uid"], freq=freq
            )

            # Handle reading history arrays
            if row["history"]:
                history_articles = row["history"].split(" ")
                session.run(
                    """
                    MATCH (s:Subscriber {id: $uid})
                    MATCH (a:Article) WHERE a.id IN $history_list
                    MERGE (s)-[:READ]->(a)
                    """,
                    uid=row["uid"], history_list=history_articles
                )

            # Generate Synthetic FOLLOWS relationship (Subscriber -> Source)
            # Users can organically follow between 0 and 3 production sources (OR CAN BE MORE)
            num_to_follow = random.randint(0, 3)
            if num_to_follow > 0 and source_ids_list:
                followed_sources = random.sample(source_ids_list, min(num_to_follow, len(source_ids_list)))
                session.run(
                    """
                    MATCH (s:Subscriber {id: $uid})
                    MATCH (src:Source) WHERE src.id IN $followed_list
                    MERGE (s)-[:FOLLOWS]->(src)
                    """,
                    uid=row["uid"], followed_list=followed_sources
                )

        # -------------------------------------------------------------
        # PHASE 3: Compute Subscriber INTERESTED_IN Topics
        # -------------------------------------------------------------
        print("Dynamically inferring Subscriber INTERESTED_IN paths...")
        # Calculates what topic categories a user reads most based on structural volume count thresholds.
        # If a subscriber reads at least 2 articles from a specific topic, they become INTERESTED_IN it.
        session.run(
            """
            MATCH (s:Subscriber)-[:READ]->(a:Article)-[:HAS_TOPIC]->(t:Topic)
            WITH s, t, count(a) AS volume
            WHERE volume >= 2
            MERGE (s)-[:INTERESTED_IN]->(t)
            """
        )

    driver.close()
    print("ETL Data Pipeline Execution Finished Successfully!")


if __name__ == "__main__":
    run_etl("./MINDsmall_train/news.tsv", "./MINDsmall_train/behaviors.tsv")

SyntaxError: invalid syntax (1970761978.py, line 1)

<h2>Graph Export Pipeline: Neo4j to CSV Archival</h2>

<p>
This pipeline exports the complete Neo4j graph structure into CSV files for external analysis, backup, and machine learning workflows.
</p>

<h3>1. Overview</h3>
<p>
The system extracts all graph nodes and relationships from Neo4j and converts them into flat tabular datasets using Pandas.
</p>

<h3>2. Node Export</h3>
<p>
The pipeline iterates through all major node types:
</p>

<ul>
  <li>Article</li>
  <li>Topic</li>
  <li>Source</li>
  <li>Entity</li>
  <li>Subscriber</li>
</ul>

<p>
Each node and its properties are retrieved from Neo4j and stored as structured CSV files.
</p>

<p>
Article embedding vectors are converted into string format to maintain readability and CSV compatibility.
</p>

<h3>3. Relationship Export</h3>
<p>
The pipeline exports graph relationships as edge mappings between node identifiers.
</p>

<ul>
  <li>HAS_TOPIC</li>
  <li>PUBLISHED</li>
  <li>FROM</li>
  <li>MENTIONS</li>
  <li>READ</li>
  <li>FOLLOWS</li>
  <li>INTERESTED_IN</li>
</ul>

<p>
Each relationship file contains:
</p>

<ul>
  <li>from_id</li>
  <li>to_id</li>
</ul>

<p>
This creates a lightweight edge-list representation of the graph structure.
</p>

<h3>4. Output</h3>
<p>
The final output consists of multiple CSV files:
</p>

<ul>
  <li>article_nodes.csv</li>
  <li>topic_nodes.csv</li>
  <li>source_nodes.csv</li>
  <li>entity_nodes.csv</li>
  <li>subscriber_nodes.csv</li>
</ul>

<p>
Relationship mappings are also exported as separate CSV files for downstream graph analytics and visualization tasks.
</p>

<h3>5. Outcome</h3>
<p>
The exported datasets provide a portable representation of the Neo4j graph and can be used for:
</p>

<ul>
  <li>data backup and archival</li>
  <li>graph visualization</li>
  <li>machine learning pipelines</li>
  <li>network analysis</li>
  <li>external reporting and research</li>
</ul>

In [3]:
Comment before executing the code (Basically it generates the CSV files of the nodes and relationships from the database.
    
def export_graph_to_csv():
    # List of node labels to pull properties for
    node_labels = ["Article", "Topic", "Source", "Entity", "Subscriber"]
    
    # List of relationship types to map mappings for
    rel_types = ["HAS_TOPIC", "PUBLISHED", "FROM", "MENTIONS", "READ", "FOLLOWS", "INTERESTED_IN"]
    
    with driver.session() as session:
        # --- EXPORT NODES ---
        for label in node_labels:
            print(f"Exporting {label} nodes to CSV...")
            query = f"MATCH (n:{label}) RETURN n"
            result = session.run(query)
            
            # Extract internal maps to create flat dictionaries
            flat_nodes = []
            for record in result:
                node = record["n"]
                properties = dict(node.items())
                # Keep vector representations readable but clean if it's an Article
                if "embedding" in properties:
                    properties["embedding"] = str(properties["embedding"])
                flat_nodes.append(properties)
            
            if flat_nodes:
                df = pd.DataFrame(flat_nodes)
                df.to_csv(f"{label.lower()}_nodes.csv", index=False)
        
        # --- EXPORT RELATIONSHIPS ---
        for rel in rel_types:
            print(f"Exporting {rel} structural connections...")
            query = f"MATCH (start)-[r:{rel}]->(end) RETURN start.id AS from_id, end.id AS to_id"
            result = session.run(query)
            
            flat_rels = [{"from_id": r["from_id"], "to_id": r["to_id"]} for r in result]
            if flat_rels:
                df = pd.DataFrame(flat_rels)
                df.to_csv(f"{rel.lower()}_relationships.csv", index=False)
                
    print("\nAll graph elements successfully archived to CSV files!")

export_graph_to_csv()

SyntaxError: invalid syntax (3091268371.py, line 1)

## Phase 2: Queries
Below are the implementations of some of the recommendation and intelligence discovery queries.

## Queries 1: Similarity

## 1.1. Graph-Based Overlap (No Vectors)
This pipeline recommends similar articles using graph structure instead of vector embeddings.

Implementation:

- The system finds Topics and Entities connected to the target article.
- It searches for other articles sharing the same Topics or Entities.
- The number of shared connections is counted.
- Articles with the highest overlap are ranked as the most related.
- The top matching articles are returned.

In [4]:
import pandas as pd

def get_graph_overlap_recommendations(article_id):
    cypher_query = """
    MATCH (target:Article {id: $aid})-[:HAS_TOPIC|MENTIONS]->(sharedNode)
          <-[:HAS_TOPIC|MENTIONS]-(candidate:Article)

    WHERE candidate <> target

    WITH candidate, count(sharedNode) AS overlapCount

    RETURN candidate.id AS ArticleId,
           candidate.title AS Title,
           overlapCount AS SharedConnections

    ORDER BY overlapCount DESC
    LIMIT 5
    """

    with driver.session() as session:
        result = session.run(cypher_query, aid=article_id)
        df = pd.DataFrame([dict(record) for record in result])

        if df.empty:
            print(f"No related articles found for {article_id}")
            return None

        return df


# Example execution
get_graph_overlap_recommendations(article_id="N4321")

,ArticleId,Title,SharedConnections
0,N35046,Report: Redskins telling teams Trent Williams ...,4
1,N39809,Redskins Rumors: If Bruce Allen won't pick up ...,4
2,N14243,"Trent Williams reports to the Redskins, the ho...",4
3,N9116,Report: Trent Williams plans to report to the ...,4
4,N14655,"Trent Williams says he had cancer, alleges Red...",3


## Create Vector Index
(Run this before the next query)

This step creates a vector index on Article embeddings in Neo4j to enable fast semantic similarity search.

Implementation:

- The index is created on the embedding property of Article nodes.
- Each embedding vector has 384 dimensions.
- Cosine similarity is used to measure semantic similarity between articles.
- The vector index allows efficient nearest-neighbor search for recommendation systems and semantic retrieval.

In [5]:
def create_vector_index():
    cypher = """
    CREATE VECTOR INDEX article_embeddings_idx IF NOT EXISTS
    FOR (a:Article)
    ON (a.embedding)
    OPTIONS {
        indexConfig: {
            `vector.dimensions`: 384,
            `vector.similarity_function`: 'cosine'
        }
    }
    """

    with driver.session() as session:
        session.run(cypher)

    print("Vector index created successfully (or already exists).")


# Run index creation
create_vector_index()

Vector index created successfully (or already exists).


## 1.2 Vector Similarity Recommendations
This step performs semantic similarity search using Neo4j’s vector index to find articles related to a given target article.

Implementation:

- The system retrieves the embedding of the target Article node.
- It queries the pre-built vector index (article_embeddings_idx) to find nearest neighbors based on embedding similarity.
- Cosine similarity is used to measure how semantically close articles are.
- The target article is excluded from the results to avoid self-matching.
- The final output returns the most similar articles ranked by similarity score, including their metadata (id, title, content).

In [6]:
def vector_similarity_recommendations(article_id, top_k=5):
    cypher_query = """
    MATCH (target:Article {id: $aid})
    CALL db.index.vector.queryNodes('article_embeddings_idx', 6, target.embedding)
    YIELD node AS similarArticle, score

    WHERE similarArticle <> target

    RETURN similarArticle.id AS ArticleId,
           similarArticle.title AS Title,
           similarArticle.content AS Content,
           score AS SimilarityScore

    ORDER BY score DESC
    LIMIT $limit
    """

    with driver.session() as session:
        result = session.run(cypher_query, aid=article_id, limit=top_k)
        df = pd.DataFrame([dict(record) for record in result])
        return df


# Example run
vector_similarity_recommendations(article_id="N4321")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=3, column=5, offset=43>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 43, 'line': 3, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    MATCH (target:Article {id: $aid})\n    CALL db.index.vector.queryNodes('article_embeddings_idx', 6, target.embedding)\n    YIELD node AS similarArticle, score\n\n    WHERE similarArticle <> target\n\n    RETURN similarArticle.id AS ArticleId,\n           similarArticle.title AS Title,\n           similarArticle.content AS Content,\n 

,ArticleId,Title,Content,SimilarityScore
0,N14243,"Trent Williams reports to the Redskins, the ho...",Your move Bruce,1.000000
1,N9116,Report: Trent Williams plans to report to the ...,Bruce vs Trent continues,0.859772
2,N36068,"AP source: Trent Williams ends holdout, report...",A person with knowledge of the situation says ...,0.829675
3,N53280,"Trent Williams has ended his holdout, is back ...",The Redskins did not find a trade target for o...,0.828137
4,N15087,Trent Williams to Cleveland rumor emerges,Dysfunctional teams do dysfunctional things. B...,0.810509


### Query 2: Hybrid Vector + Knowledge Graph Collaborative Recommendation
This pipeline generates hybrid news recommendations for a subscriber using semantic similarity and personalization signals.

Implementation:

- The system first finds the most recently read article of the user.
- It then performs vector similarity search using the article embedding and Neo4j vector index.
- Similar articles already read by the user are excluded.
- Additional recommendation boosts are applied:
    - +0.3 if the article comes from a followed source
    - +0.2 if the article belongs to a topic the user is interested in
- A final hybrid score is computed by combining semantic similarity and personalization boosts.
- The top-ranked articles are returned as personalized recommendations.

In [7]:
def get_hybrid_recommendations(user_id):
    cypher_recommend = """
    MATCH (u:Subscriber {id: $uid})-[:READ]->(last:Article)
    WITH u, last
    ORDER BY last.published_at DESC
    LIMIT 1

    CALL db.index.vector.queryNodes('article_embeddings_idx', 20, last.embedding)
    YIELD node AS candidate, score
    WHERE candidate <> last AND NOT (u)-[:READ]->(candidate)

    OPTIONAL MATCH (u)-[:FOLLOWS]->(src:Source)<-[:FROM]-(candidate)
    OPTIONAL MATCH (u)-[:INTERESTED_IN]->(t:Topic)<-[:HAS_TOPIC]-(candidate)

    WITH candidate, score,
         CASE WHEN src IS NOT NULL THEN 0.3 ELSE 0.0 END AS sourceBoost,
         CASE WHEN t IS NOT NULL THEN 0.2 ELSE 0.0 END AS topicBoost

    WITH candidate, (score + sourceBoost + topicBoost) AS FinalScore
    RETURN candidate.id AS ArticleId, 
           candidate.title AS Title, 
           candidate.url AS URL,
           round(FinalScore, 4) AS SimilarityScore
    ORDER BY FinalScore DESC
    LIMIT 5;
    """
    
    with driver.session() as session:
        res = session.run(cypher_recommend, uid=user_id)
        # Parse straight into a data frame map for display
        df = pd.DataFrame([dict(record) for record in res])
        return df

# Test execution (Replace with a valid UID from your behaviors dataset partition)
get_hybrid_recommendations(user_id="U19739")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=7, column=5, offset=131>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 131, 'line': 7, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    MATCH (u:Subscriber {id: $uid})-[:READ]->(last:Article)\n    WITH u, last\n    ORDER BY last.published_at DESC\n    LIMIT 1\n\n    CALL db.index.vector.queryNodes('article_embeddings_idx', 20, last.embedding)\n    YIELD node AS candidate, score\n    WHERE candidate <> last AND NOT (u)-[:READ]->(candidate)\n\n    OPTIONAL MATCH (u)-

,ArticleId,Title,URL,SimilarityScore
0,N39326,2 Arizona families seek missing loved ones amo...,https://assets.msn.com/labs/mind/AAJAerB.html,1.2908
1,N5550,Search For Missing Florida Girl Finds Human Re...,https://assets.msn.com/labs/mind/BBWEYPA.html,1.2682
2,N8258,New Hampshire Couple's Adventure Ends in Myste...,https://assets.msn.com/labs/mind/AAJRz07.html,1.1053
3,N52428,Texas police hunt for killer after finding New...,https://assets.msn.com/labs/mind/AAJNBpD.html,1.0971
4,N19748,"Missing NH Couple Confirmed Dead In Texas, Pol...",https://assets.msn.com/labs/mind/AAJIyQh.html,1.0965


### Query 3: Real-time Breaking News Topic Trend Detection
This pipeline detects trending news topics based on user reading activity in the graph database.

Implementation:

- The system traverses Subscriber → Article → Topic relationships.
- It counts:
    - the number of unique articles in each topic
    - the total number of reader interactions
- A trend velocity score is computed using:
    - TrendVelocity = TotalReads / ArticleCount
- Topics with:
    - more than 2 articles
    - and high reading momentum (> 1.5),
    are considered trending.

In [8]:
def detect_trending_news_topics():
    cypher_trending = """
    MATCH (s:Subscriber)-[:READ]->(a:Article)-[:HAS_TOPIC]->(t:Topic)
    WITH t, 
         count(distinct a) AS ArticleCount, 
         count(s) AS TotalReads

    WITH t, ArticleCount, TotalReads,
         (toFloat(TotalReads) / toFloat(ArticleCount)) AS TrendVelocity
         
    WHERE ArticleCount > 2 AND TrendVelocity > 1.5
    RETURN t.name AS TrendingTopic, 
           ArticleCount AS UniqueArticlesInTopic, 
           TotalReads AS RapidReaderVolume,
           round(TrendVelocity, 2) AS MomentumScore
    ORDER BY MomentumScore DESC
    LIMIT 5;
    """
    
    with driver.session() as session:
        res = session.run(cypher_trending)
        df = pd.DataFrame([dict(record) for record in res])
        return df

# Run engine analytics diagnostics
detect_trending_news_topics()

,TrendingTopic,UniqueArticlesInTopic,RapidReaderVolume,MomentumScore
0,tv,690,71064,102.99
1,entertainment,424,31644,74.63
2,movies,475,34580,72.80
3,lifestyle,1642,97797,59.56
4,music,482,24401,50.62


## Query4: Engagement: The "Echo Chamber" Breaker (Serendipity Discovery)

This step generates serendipity-based news recommendations designed to break filter bubbles and improve content diversity in user engagement.

Implementation:

- The system first identifies topics the user is interested in via INTERESTED_IN relationships.
- It extracts entities the user has previously encountered from articles within those topics.
- It then searches for new articles that mention the same entities but have not been read by the user.
- To ensure novelty, it filters out articles from sources the user already follows.
- Finally, it introduces randomness in ranking to promote serendipitous discovery rather than purely similarity-based recommendations,returning articles from unfamiliar publishers along with overlapping entities.

In [9]:
import pandas as pd

def serendipity_recommendations(user_id):
    cypher_query = """
    MATCH (u:Subscriber {id: $uid})-[:INTERESTED_IN]->(t:Topic)

    // Entities user has encountered in this topic
    MATCH (u)-[:READ]->(past:Article)-[:MENTIONS]->(e:Entity)
    WHERE (past)-[:HAS_TOPIC]->(t)

    // Candidate articles sharing same entities in same topic
    MATCH (candidate:Article)-[:MENTIONS]->(e)
    WHERE (candidate)-[:HAS_TOPIC]->(t)
      AND NOT (u)-[:READ]->(candidate)

    // Prefer unseen publishers
    MATCH (candidate)-[:FROM]->(altSource:Source)
    WHERE NOT (u)-[:FOLLOWS]->(altSource)

    RETURN candidate.id AS SurpriseArticleId,
           candidate.title AS Title,
           altSource.name AS UnfamiliarPublisher,
           collect(distinct e.name)[0..3] AS OverlappingEntities
    ORDER BY rand()
    LIMIT 5
    """

    with driver.session() as session:
        result = session.run(cypher_query, uid=user_id)
        df = pd.DataFrame([dict(record) for record in result])
        return df


# Example run
serendipity_recommendations(user_id="U13740")

,SurpriseArticleId,Title,UnfamiliarPublisher,OverlappingEntities
0,N59138,Trump impeachment hearings: 5 key takeaways fr...,Reuters,[Joe Biden]
1,N55362,Memorial Park Golf Course Renovation Phase I C...,Reuters,[Houston Astros]
2,N51250,'Code Black' security threat at Hartsfield-Jac...,Reuters,[American Airlines]
3,N17285,OSHA fines Carowinds parent company after work...,Bloomberg,[South Carolina]
4,N27894,Activists press AZ senators to fund election s...,ESPN,[United States Senate]


## Query 5: Entity Graph Analytics: Co-occurrence Mapping (Hidden Connections

This query analyzes entity co-occurrence patterns within news articles using the Neo4j graph database.

Implementation:

- The system first finds articles that mention the target entity.
- It then searches for other entities mentioned in the same articles.
- The number of shared article mentions is counted for each co-occurring entity.
- Entities are ranked based on co-mention frequency.
- The top related entities, their types, and co-occurrence counts are returned.

This helps identify semantic relationships and commonly associated entities in news content.

In [10]:
def analyze_entity_co_occurrence(target_entity_id):
    cypher_co_occurrence = """
    // Match the target entity of interest
    MATCH (target:Entity {id: $entity_id})<-[:MENTIONS]-(a:Article)

    // Find other entities mentioned alongside it in the exact same articles
    MATCH (a)-[:MENTIONS]->(coOccurring:Entity)
    WHERE coOccurring <> target

    WITH coOccurring, count(a) AS SharedArticles
    ORDER BY SharedArticles DESC
    LIMIT 10

    // Return the structural context map of relationships
    RETURN coOccurring.name AS RelatedEntity,
           coOccurring.type AS EntityType,
           SharedArticles AS NumberOfCoMentions;
    """
    
    # Execute the query using the established driver session
    with driver.session() as session:
        res = session.run(cypher_co_occurrence, entity_id=target_entity_id)
        df = pd.DataFrame([dict(record) for record in res])
        if df.empty:
            print(f"No co-occurring entities found for Entity ID: {target_entity_id}")
            return None      
        return df

# Run the intelligence query for the requested target (e.g., Q43274)
analyze_entity_co_occurrence(target_entity_id="Q80976")

,RelatedEntity,EntityType,NumberOfCoMentions
0,"Charles, Prince of Wales",P,1
1,Elizabeth II,P,1


## Query 6: Network Optimization: Influencer Source Hub Filtering
This step analyzes cross-publisher user movement to identify influential news sources acting as “hub” nodes in the reading network.

Implementation:

- The system tracks user reading sequences across different sources.
- It detects transitions where a user reads from one source and later switches to another.
- Only valid transitions are considered by ensuring the second article is published after the first.
- It counts how many distinct users move between each pair of sources.
- Finally, it identifies publisher pairs with strong reader migration flow, representing cross-pollination and influence in the news ecosystem.

In [12]:
def publisher_hub_analysis():
    cypher_query = """
    MATCH (s:Subscriber)-[:READ]->(a1:Article)-[:FROM]->(src1:Source)
    MATCH (s)-[:READ]->(a2:Article)-[:FROM]->(src2:Source)
    WHERE src1 <> src2
      AND a1.published_at < a2.published_at

    WITH src1, src2, count(distinct s) AS MigratedUsers
    WHERE MigratedUsers > 1

    RETURN src1.name AS FromPublisher,
           src2.name AS ToPublisherBridge,
           MigratedUsers AS ReaderTrafficFlow

    ORDER BY ReaderTrafficFlow DESC
    LIMIT 5
    """

    with driver.session() as session:
        result = session.run(cypher_query)
        df = pd.DataFrame([dict(record) for record in result])
        return df


# Example run
publisher_hub_analysis()

,FromPublisher,ToPublisherBridge,ReaderTrafficFlow
0,CNN,Reuters,24538
1,Reuters,CNN,24095
2,Reuters,ESPN,24002
3,CNN,ESPN,23903
4,The New York Times,Reuters,23876


<h2>Entity Enrichment Pipeline</h2>

<p>
This pipeline enriches Entity nodes in Neo4j using live data fetched incrementally from the Wikidata API.
</p>

<h3>1. Overview</h3>
<p>
The system checks whether an entity already contains enrichment data. If not, it queries Wikidata, extracts structured metadata, and updates the local Neo4j graph.
</p>

<h3>2. Eligibility Check</h3>
<p>
The pipeline first verifies whether the entity exists and checks its enrichment status.
</p>

<ul>
  <li>If already enriched → fetch from local graph</li>
  <li>If not enriched → trigger Wikidata API enrichment</li>
</ul>

<h3>3. Wikidata Enrichment</h3>
<p>
The pipeline incrementally retrieves official metadata from Wikidata.
</p>

<p>
Extracted properties include:
</p>

<ul>
  <li>Description</li>
  <li>Official website</li>
  <li>Instance type (P31)</li>
  <li>Industries (P452)</li>
  <li>Country (P17)</li>
</ul>

<p>
Safe parsing and fallback defaults are used to handle missing or incomplete fields.
</p>

<h3>4. Neo4j Upsert</h3>
<p>
The enriched metadata is written back into the Entity node inside Neo4j.
</p>

<p>
Additional pipeline control fields are also maintained:
</p>

<ul>
  <li>enrichment_status</li>
  <li>last_enriched</li>
</ul>

<h3>5. Outcome</h3>
<p>
The final graph contains enriched entities connected with structured external knowledge, enabling better semantic analysis, recommendations, and knowledge graph exploration.
</p>

In [13]:
import requests
from datetime import datetime
import pandas as pd

def enrich_entity_on_demand(entity_id):
    """
    Checks if an Entity needs enrichment, fetches official data from the 
    Wikidata JSON API incrementally, and upserts it back to Neo4j cleanly.
    """
    
    # --- PHASE 1: CHECK ELIGIBILITY IN NEO4J ---
    check_query = """
    MATCH (e:Entity {id: $eid})
    RETURN e.enrichment_status AS status
    """
    with driver.session() as session:
        result = session.run(check_query, eid=entity_id).single()
        if not result:
            print(f"Entity {entity_id} does not exist in your local Neo4j database.")
            return None
        
        # If already successfully enriched, return the current node state from Neo4j
        if result["status"] == "done":
            print(f"Entity {entity_id} is already enriched. Fetching from local graph...")
            return fetch_local_entity(entity_id)

    # --- PHASE 2: FETCH INCREMENTALLY FROM WIKIDATA API ---
    print(f"Incremental Fetch: Querying Wikidata API for {entity_id}...")
    api_url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "ids": entity_id,
        "format": "json",
        "props": "claims|descriptions",
        "languages": "en"
    }
    
    # Initialize pipeline control baselines
    enriched_data = {
        "description": "None",
        "website": "None",
        "instance_of": "Unknown",
        "industries": [],
        "country": "None",
        "enrichment_status": "failed",
        "last_enriched": datetime.now().isoformat()
    }
    
    try:
        response = requests.get(api_url, params=params, headers={"User-Agent": "NewsIntelligenceBot/1.0"}, timeout=5)
        response.raise_for_status()
        data = response.json()
        
        entity_info = data.get("entities", {}).get(entity_id, {})
        
        if "missing" in entity_info:
            raise ValueError(f"Entity {entity_id} not found on Wikidata.")

        # 1. Safe parsing for English Description
        enriched_data["description"] = entity_info.get("descriptions", {}).get("en", {}).get("value", "None")

        claims = entity_info.get("claims", {})

        # Helper internal function to cleanly unpack primary statement targets
        def get_claim_numeric_id(property_code):
            claim_list = claims.get(property_code, [])
            if claim_list:
                mainsnak = claim_list[0].get("mainsnak", {})
                datavalue = mainsnak.get("datavalue", {})
                return f"Q{datavalue.get('value', {}).get('numeric-id', '')}"
            return "None"

        # 2. Get Instance Of (P31) - e.g. human, enterprise
        instance_q = get_claim_numeric_id("P31")
        if instance_q != "Q": enriched_data["instance_of"] = instance_q

        # 3. Get Official Website (P856)
        web_claims = claims.get("P856", [])
        if web_claims:
            enriched_data["website"] = web_claims[0].get("mainsnak", {}).get("datavalue", {}).get("value", "None")

        # 4. Get Industries List (P452)
        industry_claims = claims.get("P452", [])
        for ind in industry_claims:
            ind_id = ind.get("mainsnak", {}).get("datavalue", {}).get("value", {}).get("numeric-id")
            if ind_id:
                enriched_data["industries"].append(f"Q{ind_id}")

        # 5. Get Country (P17)
        country_q = get_claim_numeric_id("P17")
        if country_q != "Q": enriched_data["country"] = country_q

        # If it reached here without crashing, the payload is successfully compiled
        enriched_data["enrichment_status"] = "done"

    except Exception as e:
        print(f"Enrichment pipeline halted for {entity_id}: {e}")
        enriched_data["enrichment_status"] = "failed"

    # --- PHASE 3: UPSERT BACK TO NEO4J ---
    upsert_query = """
    MATCH (e:Entity {id: $eid})
    SET e.description = $desc,
        e.website = $web,
        e.instance_of = $iof,
        e.industries = $inds,
        e.country = $cty,
        e.enrichment_status = $status,
        e.last_enriched = datetime($ts)
    RETURN e.id AS id, e.name AS name, e.description AS description, e.website AS website, 
           e.instance_of AS instance_of, e.industries AS industries, e.country AS country, e.enrichment_status AS enrichment_status
    """
    
    with driver.session() as session:
        res = session.run(
            upsert_query,
            eid=entity_id,
            desc=enriched_data["description"],
            web=enriched_data["website"],
            iof=enriched_data["instance_of"],
            inds=enriched_data["industries"],
            cty=enriched_data["country"],
            status=enriched_data["enrichment_status"],
            ts=enriched_data["last_enriched"]
        )
        return pd.DataFrame([dict(res.single())])


def fetch_local_entity(entity_id):
    """Fallback helper to fetch a clean dataframe directly from local storage."""
    query = """
    MATCH (e:Entity {id: $eid})
    RETURN e.id AS id, e.name AS name, e.description AS description, e.website AS website, 
           e.instance_of AS instance_of, e.industries AS industries, e.country AS country, e.enrichment_status AS enrichment_status
    """
    with driver.session() as session:
        return pd.DataFrame([dict(session.run(query, eid=entity_id).single())])

In [14]:
df_first_run = enrich_entity_on_demand("Q2283")
display(df_first_run)

Entity Q2283 is already enriched. Fetching from local graph...


,id,name,description,website,instance_of,industries,country,enrichment_status
0,Q2283,Microsoft,American multinational technology corporation,https://www.microsoft.com/,Q1058914,"[Q21157865, Q880371, Q638608, Q1666934, Q73768...",Q30,done


In [15]:
df_second_run = enrich_entity_on_demand("Q2283")
display(df_second_run)

Entity Q2283 is already enriched. Fetching from local graph...


,id,name,description,website,instance_of,industries,country,enrichment_status
0,Q2283,Microsoft,American multinational technology corporation,https://www.microsoft.com/,Q1058914,"[Q21157865, Q880371, Q638608, Q1666934, Q73768...",Q30,done


In [16]:
Comment this line
driver.close()
print("Graph connection safely terminated.")

SyntaxError: invalid syntax (1222708072.py, line 1)